In [29]:
import numpy as np
import pandas as pd
%load_ext autoreload
%autoreload 2

from autobound.causalProblem import causalProblem
from autobound.DAG import DAG
import io

$\mathcal{G}:$
```
          [R₃]         [R₁]     [R₂]
        ↙    ↘       ↙    ↘  ↙    ↘
       A ───→ B ───→ X ────→ M ────→ Y 
              ↑       │      ↑
              ┊       ↓      │
             [R₄]────→C─────→D
```

We have $\theta=P(Y=1|do(X=1,M=1))$.

The mission of this notebook:
Calculate 
1. $pot(Y,M|do(X=1))$
2. $pot(Y,M|do(X=0))$


## Step 0: Simulate SCM

Remark: W.l.o.g. We can assume a simpler graph
$X \to M \to Y$

In [27]:
N_samples = 100000
X = np.random.binomial(1, 0.5, size=N_samples)
# M is the inverse of X 70% of the time, and the same as X 30% of the time
M = np.where(np.random.rand(N_samples) < 0.7, 1 - X, X)
# is the inverse of M 70% of the time, and the same as M 30% of the time
Y = np.where(np.random.rand(N_samples) < 0.7, 1 - M, M)

# compute probabilities for every combination of values (000, ... 111) and safe them as csv: X,M,Y,prob
data = []
for x in [0,1]:
    for m in [0,1]:
        for y in [0,1]:
            prob = np.mean((X == x) & (M == m) & (Y == y))
            data.append({'X': x, 'M': m, 'Y': y, 'prob': prob})
df = pd.DataFrame(data)
df.to_csv('data/figure-4-obs.csv', index=False)


## Step 1: Compute $\mathcal{W}_\emptyset$

In [62]:
dag = DAG()
dag.from_structure("X -> M, M -> Y, U->X, U->M, U->Y", unob='U')
problem = causalProblem(dag)
problem.load_data('data/figure-4-obs.csv')
problem.add_prob_constraints()
problem.set_estimand(problem.query('Y(X=1, M=1)=1'))
program = problem.write_program()
program.to_pip('figure4.pip')

(lb, ub) = program.run_pyomo('ipopt', verbose=False)   # or 'couenne' if you prefer
calW_empty = ub-lb

In [61]:
calW_empty

0.4995301189651238